# ASEAN: raw SDMX to analysis panel
Run all cells to reproduce processed outputs from the latest immutable raw snapshot. Download separately with `python scripts/asean_pipeline.py fetch`. No network is required here.
See `reports/qa_report.md` for actual coverage and pending analyst acceptance.

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'scripts/asean_pipeline.py').exists():
    ROOT = ROOT.parent
spec = importlib.util.spec_from_file_location('pipeline', ROOT / 'scripts/asean_pipeline.py')
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)
pipeline.build()

## QA gates
No unexplained duplicate country–indicator–year keys. Invalid numeric values and unit mismatches are quarantined. Missing values are not filled. Annual lag uses exact calendar year. Unknown/estimated/forecast inputs are not considered observed.

In [ ]:
panel = pd.read_csv(ROOT / 'data/processed/asean_comparison.csv')
coverage = pd.read_csv(ROOT / 'reports/coverage_matrix.csv')
assert not panel.duplicated(['country','year']).any()
assert pd.read_csv(ROOT / 'reports/rejected_rows.csv').empty
display(panel.head())
display(coverage.groupby(['country','indicator']).available.agg(['sum','count']))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(ROOT/'tests'), '-p', 'test_asean_pipeline.py'], check=True)
print((ROOT/'reports/cross_check.md').read_text(encoding='utf-8'))

In [ ]:
print((ROOT/'reports/qa_report.md').read_text(encoding='utf-8'))
display(pd.read_csv(ROOT/'metadata/data_dictionary.csv'))